# Yash — Tau2 7B — Team-Aligned Strands Classifier

This is a brand-new Yash notebook. Teammate notebooks and the earlier recorded panels remain unchanged.

The primary head is **seed zero, balanced BCE on strict answer exact-match**, using the continuous Strands score. Its exact bytes and all cohort IDs are frozen below before fresh calls. This is a different objective from the matched PG retraining, which retained the immediate-stop failure and is reported separately. All five heads were frozen before cached test scoring, but the cached test80 had already been viewed and is exploratory, not a new blind evaluation. Fresh cohorts supply the prospective test.

No credentials or paid APIs are needed. Select a GPU. Approve any required Colab presence/Drive checks yourself. Keep failures in the denominators; running the notebook is not a promise of positive results.

This optional engineering variant uses pinned **Qwen2.5-7B, NF4 weights and FP16 computation**. Official Tau tools and evaluators remain unchanged; local customers/judges use the same 7B model. The QA-trained head is uncalibrated for retail actions, so these results are not leaderboard comparable. Two development tasks (15,62; eight episodes) stay separate from ten fresh evaluation tasks (forty episodes).

## 1. Install the exact source
This checkout belongs to this new notebook. It does not edit a teammate's working files.

In [ ]:
import sys, subprocess, pathlib, os, json
SOURCE_COMMIT = '696f8063471b616a24288249d79cce0accec53e3'
ROOT = pathlib.Path("/content/Yash_HaltIQ_Aligned_20261030_Tau2")
def checked(*arguments, **kwargs):
    subprocess.run(list(arguments), check=True, **kwargs)
assert (3, 12) <= sys.version_info[:2] < (3, 14), "Use a Python 3.12/3.13 runtime"
if not ROOT.exists():
    checked("git", "clone", "--no-checkout", "https://github.com/acm-research-f26/Halt-IQ.git", str(ROOT))
checked("git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", SOURCE_COMMIT)
checked("git", "-C", str(ROOT), "checkout", "--detach", SOURCE_COMMIT)
required_modules = ['haltiq_pair/team_training.py', 'haltiq_pair/aligned_gate.py', 'haltiq_pair/aligned_hotpot.py', 'haltiq_pair/aligned_tau.py', 'haltiq_pair/strong_writer.py']
missing_modules = [name for name in required_modules if not (ROOT/name).is_file()]
assert not missing_modules, "Draft source pin is missing the new modules; regenerate with the verified integration commit: " + repr(missing_modules)
checked(sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT/"requirements-colab.txt"))
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
try:
    installed_torchao = version("torchao")
except PackageNotFoundError:
    installed_torchao = None
if installed_torchao is not None and Version(installed_torchao) < Version("0.16"):
    print("Removing unused incompatible TorchAO", installed_torchao)
    checked(sys.executable, "-m", "pip", "uninstall", "-y", "torchao")
checked(sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT))
checked(sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.50.2")
TAU_ROOT = pathlib.Path("/content/Yash_Tau2_Official_20261030")
TAU_COMMIT = 'cc1df47baa1544a34bb8c8e5c4ca72ad223b96a2'
if not TAU_ROOT.exists():
    checked("git", "clone", "--no-checkout", "https://github.com/sierra-research/tau2-bench.git", str(TAU_ROOT))
checked("git", "-C", str(TAU_ROOT), "fetch", "--depth", "1", "origin", TAU_COMMIT)
checked("git", "-C", str(TAU_ROOT), "checkout", "--detach", TAU_COMMIT)
checked(sys.executable, "-m", "pip", "install", "-q", "-e", str(TAU_ROOT))
assert version("bitsandbytes") == "0.50.2"
print("Pinned source installed:", SOURCE_COMMIT)


## 2. Create Yash's new result folder
Drive is preferred; a failed mount falls back to temporary storage. Each Run all creates a fresh timestamp/UUID directory. Download the ZIP before disconnecting if persistent=False.

In [ ]:
from google.colab import drive
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid
USE_DRIVE = True
storage_root = pathlib.Path("/content/Yash_HaltIQ_Aligned_Results_20261030")
backend, mount_error = "runtime", None
if USE_DRIVE:
    try:
        drive.mount("/content/drive")
        storage_root = pathlib.Path("/content/drive/MyDrive/Yash_HaltIQ_Pair")
        backend = "google_drive"
    except Exception as error:
        mount_error = type(error).__name__
        print("Drive connection failed; this new result folder will be temporary.")
RUN_ID = datetime.now(ZoneInfo("America/Chicago")).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
OUT = storage_root / "team-aligned-20261030" / "tau2" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
storage_metadata = dict(backend=backend, requested_drive=USE_DRIVE,
    persistent=backend == "google_drive", drive_mount_error_type=mount_error,
    requires_download_before_disconnect=backend != "google_drive", result_directory=str(OUT))
with (OUT/"storage.json").open("x", encoding="utf-8") as stream:
    json.dump(storage_metadata, stream, indent=2)
with (OUT/"environment.txt").open("x", encoding="utf-8") as stream:
    stream.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Yash's new result folder:", OUT)
if not storage_metadata["persistent"]:
    print("WARNING: download the final evidence ZIP before disconnecting this temporary runtime.")


## 3. Freeze the exact head and public cohort files
SHA256 checks preserve the original JSON bytes. Existing frozen files must match exactly. No training or threshold selection happens in this notebook.

In [ ]:
import hashlib
EMBEDDED_FILES = {'Yash-BCE-Strands-Head-seed0.json': {'sha256': 'b0d5256f6538898d09a0a07322124def66b2abe505a08abf92c13f81fd9dcd0b', 'text': '{\r\n  "schema": "yash-retrained-strands-feature-stop-head-v1",\r\n  "features": [\r\n    "distance",\r\n    "round_fraction",\r\n    "distance_delta",\r\n    "strands_probability",\r\n    "has_previous_draft"\r\n  ],\r\n  "weights": [\r\n    0.014457383193075657,\r\n    -0.04416543245315552,\r\n    -0.6141678690910339,\r\n    1.3763855695724487,\r\n    0.15424859523773193\r\n  ],\r\n  "bias": -1.0333242416381836,\r\n  "seed": 0,\r\n  "stop_threshold": 0.5,\r\n  "metadata": {\r\n    "objective": "balanced_BCE_exact_match",\r\n    "source_sri_revision": "5503cabeb2ec0094413b7d491ccb08ec9c991c61",\r\n    "source_team_revision": "6c72991652400dcfec8f464a331efb95d5d1cf8b",\r\n    "joined_csv_sha256": "15edd9ea608e231ccd8f74fe461f2b5f917f80187140a4edb303fc1bf93ce394",\r\n    "source_questions_sha256": "c82fb64d47e75c1b50938b3d8be2d20c364273cb4404171283397c5efaa81dc2",\r\n    "source_protocol_sha256": "386fb3667fff2ac810149d5e327fe3a0cf55089477560d8f2ddeca512a8e4ccc",\r\n    "training_module_sha256": "f9cf1fc20f4618c67582e9a30f1d46938fa55b9214541a17b9962f68fe36783a",\r\n    "feature_module_sha256": "70d9288c720268f319f6758bbae6fea8dc67a36a88202c5ba345eef8d2d9ac33",\r\n    "feature_encoder": "same public lexical/history features plus continuous Strands probability",\r\n    "label": "strict exact-match em column, training labels only for optimizer; dev labels only for checkpoint selection",\r\n    "no_language_model_training": true,\r\n    "live_seed_predeclared": 0,\r\n    "software": {\r\n      "python": "3.12.10",\r\n      "torch": "2.7.1+cu128"\r\n    },\r\n    "trained": true,\r\n    "selected_epoch": 200,\r\n    "best_dev_balanced_bce": 0.7020193338394165,\r\n    "train_balance": {\r\n      "positive_rounds": 72,\r\n      "negative_rounds": 228,\r\n      "pos_weight": 3.1666666666666665,\r\n      "source": "training partition only",\r\n      "question_round_weight": 0.16666666666666666,\r\n      "normalization": "mean of per-question mean losses"\r\n    },\r\n    "training_config": {\r\n      "epochs": 200,\r\n      "learning_rate": 0.01,\r\n      "l2": 0.01\r\n    },\r\n    "train_ids": [\r\n      "hotpot_5a711ef85542994082a3e5ac",\r\n      "hotpot_5a714de95542994082a3e7a8",\r\n      "hotpot_5a7302b75542991f9a20c5fd",\r\n      "hotpot_5a7363ec5542991f29ee2dd7",\r\n      "hotpot_5a745a6055429974ef308be4",\r\n      "hotpot_5a79b495554299148911fa3b",\r\n      "hotpot_5a7a11c85542996a35c170b4",\r\n      "hotpot_5a7a276d5542996c55b2dd27",\r\n      "hotpot_5a7a3a2a5542996c55b2dd41",\r\n      "hotpot_5a7a5ccf55429941d65f25d7",\r\n      "hotpot_5a7cd0fd55429909bec76823",\r\n      "hotpot_5a7d2aa05542995f4f40221f",\r\n      "hotpot_5a7eacd855429930675135c3",\r\n      "hotpot_5a7f62405542992e7d278cf0",\r\n      "hotpot_5a8078d85542995d8a8ddf78",\r\n      "hotpot_5a807e6f5542995d8a8ddf80",\r\n      "hotpot_5a808c215542995d8a8ddf90",\r\n      "hotpot_5a82817555429954d2e2eb5a",\r\n      "hotpot_5a83c2e25542996488c2e4bc",\r\n      "hotpot_5a844fd35542990548d0b274",\r\n      "hotpot_5a8483b45542992a431d1a40",\r\n      "hotpot_5a85fe2f5542996432c57165",\r\n      "hotpot_5a87bada5542994846c1cddc",\r\n      "hotpot_5a882ffb5542994846c1ce0d",\r\n      "hotpot_5a8b4a5f55429949d91db531",\r\n      "hotpot_5a8b7d925542995d1e6f13bd",\r\n      "hotpot_5a8bb7f95542996e8ac889ff",\r\n      "hotpot_5a8da1815542994ba4e3dcd7",\r\n      "hotpot_5ab2c6d955429916697740c3",\r\n      "hotpot_5ab553e2554299494045ef81",\r\n      "hotpot_5ab6d9b25542995eadef00c8",\r\n      "hotpot_5aba71be5542994dbf01990c",\r\n      "hotpot_5abdf12255429976d4830a2f",\r\n      "hotpot_5ac2a399554299218029dada",\r\n      "hotpot_5ac50bca55429924173fb57f",\r\n      "hotpot_5adc0f8d55429947ff1738ef",\r\n      "hotpot_5adc368155429944faac2468",\r\n      "hotpot_5adcf20f5542994ed6169c2c",\r\n      "hotpot_5adf32ee5542995534e8c767",\r\n      "hotpot_5adfe37c554299025d62a377",\r\n      "hotpot_5adffc8d55429925eb1afbc6",\r\n      "hotpot_5ae1681b554299422ee9967c",\r\n      "hotpot_5ae1a7c8554299234fd042b0",\r\n      "hotpot_5ae25e835542992decbdccd1",\r\n      "hotpot_5ae268f35542994d89d5b416",\r\n      "hotpot_5ae2c4d75542991a06ce98d0",\r\n      "hotpot_5ae377c45542990afbd1e16b",\r\n      "hotpot_5ae4bbf05542995ad6573e32",\r\n      "hotpot_5ae5305b5542990ba0bbb1f5",\r\n      "hotpot_5ae5fb55554299546bf82ff6"\r\n    ],\r\n    "dev_ids": [\r\n      "hotpot_5a7149595542994082a3e76f",\r\n      "hotpot_5a7577145542996c70cfaf03",\r\n      "hotpot_5a7b74485542997c3ec97190",\r\n      "hotpot_5a7b820c5542997c3ec971d8",\r\n      "hotpot_5a7d9d2b5542997cc2c47464",\r\n      "hotpot_5a7def0d5542990b8f503af9",\r\n      "hotpot_5a7f5e0455429969796c1a12",\r\n      "hotpot_5a80b8845542992bc0c4a7de",\r\n      "hotpot_5a8549c65542997b5ce3ffe6",\r\n      "hotpot_5a90937955429933b8a20568",\r\n      "hotpot_5a90b7f355429916514e752d",\r\n      "hotpot_5ab472a35542991779162c98",\r\n      "hotpot_5ab520dd5542996a3a96a02c",\r\n      "hotpot_5abd27ba5542993062266ba7",\r\n      "hotpot_5abffc0d5542990832d3a1e2",\r\n      "hotpot_5ac1b1345542994ab5c67dd0",\r\n      "hotpot_5add7ce55542992ae4cec585",\r\n      "hotpot_5ae3a8985542992f92d82329",\r\n      "hotpot_5ae4c8f35542990ba0bbb130",\r\n      "hotpot_5ae5caff554299546bf82f69"\r\n    ],\r\n    "selected_parameter_l2_from_initial": 2.118232488632202,\r\n    "selection": "strict minimum dev balanced BCE using training class ratio; earliest epoch wins ties"\r\n  },\r\n  "payload_sha256": "32de88a42a63d066f164a693a033982de9fa101323c0bf74776813b23fd016dc"\r\n}\r\n'}, 'Yash-Tau-Smoke-IDs.json': {'sha256': '82198331dfb3c89049c400bbc802de3003149a78a63dcb8584d7356befc797eb', 'text': '{\r\n  "task_ids": [\r\n    "15",\r\n    "62"\r\n  ]\r\n}'}, 'Yash-Tau-Evaluation-IDs.json': {'sha256': '728060fd6398c0080955c9ba3ffe38db69e9f8b8b031bdb592055d14eee45d6d', 'text': '{\r\n  "task_ids": [\r\n    "107",\r\n    "89",\r\n    "83",\r\n    "109",\r\n    "23",\r\n    "37",\r\n    "79",\r\n    "68",\r\n    "34",\r\n    "81"\r\n  ]\r\n}'}}
FROZEN = OUT/"frozen-inputs"
FROZEN.mkdir(exist_ok=True)
for name, item in EMBEDDED_FILES.items():
    raw = item["text"].encode("utf-8")
    assert hashlib.sha256(raw).hexdigest() == item["sha256"], "Embedded input bytes changed"
    path = FROZEN/name
    if path.exists():
        assert path.read_bytes() == raw, "Existing frozen input changed; choose a new OUT"
    else:
        with path.open("xb") as stream:
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
HEAD = FROZEN/"Yash-BCE-Strands-Head-seed0.json"
assert hashlib.sha256(HEAD.read_bytes()).hexdigest() == 'b0d5256f6538898d09a0a07322124def66b2abe505a08abf92c13f81fd9dcd0b'
from haltiq_pair.team_training import StrandsHead
selected_head = StrandsHead.from_json(HEAD)
assert selected_head.seed == 0
assert selected_head.metadata["objective"] == "balanced_BCE_exact_match"
plan = dict(source_commit=SOURCE_COMMIT, benchmark='Tau2',
    frozen_input_sha256={name: item["sha256"] for name, item in EMBEDDED_FILES.items()},
    primary_head_objective="balanced_BCE_exact_match", deployment_seed_predeclared=0,
    previously_viewed_cached_test80="exploratory, not a fresh blind evaluation",
    PG_negative_result="matched PG retraining stopped at round one; retained separately",
    frozen_before_fresh_model_calls=True)

plan_path = OUT/"notebook-plan.json"
if plan_path.exists():
    assert json.loads(plan_path.read_text()) == plan
else:
    with plan_path.open("x", encoding="utf-8") as stream:
        json.dump(plan, stream, indent=2)
print("Exact frozen inputs verified:", plan["frozen_input_sha256"])


## 4. Run software controls
These tests use scripted fixtures and are not fresh benchmark results.

In [ ]:
test_modules = ["tests_pair.test_strong_writer", "tests_pair.test_aligned_tau"]
test_env = dict(os.environ, LOGURU_LEVEL="ERROR")
test_env["HALTIQ_TAU_UPSTREAM"] = str(TAU_ROOT)
checked(sys.executable, "-B", "-m", "unittest", *test_modules, "-q", cwd=ROOT, env=test_env)
checked("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")


## 5. Qualify the actual GPU wiring
The invented public fact exercises AWS Strands writer/reviewer calls, output parsing and the pinned verifier. This is engineering evidence, not a benchmark score.

In [ ]:
engineering_script = 'import json, sys\nfrom pathlib import Path\nfrom haltiq_pair.strong_writer import StrongRuntime, MODEL, validate_health\nfrom haltiq_pair.tau import parse_proposal\nfrom haltiq_pair.team_training import StrandsHead\nout, head_path = Path(sys.argv[1]), Path(sys.argv[2])\nassert not out.exists(), "Preserve earlier engineering qualifications"\nwith StrongRuntime(out, StrandsHead.from_json(head_path)) as (agents, gate):\n    prompt = \'Public fact: Paris is the capital of France. Propose the next assistant text answering the capital. Return exactly JSON with content (nonempty string) and tool_calls ([]).\'\n    raw = agents.generate(\'writer\', \'Propose only the next assistant action. No tool has executed.\', prompt)\n    proposal = parse_proposal(raw, set())\n    review = agents.generate(\'reviewer\', \'Use only the supplied public fact. Give concise feedback.\', prompt + \'\\nProposed action:\\n\' + raw)\n    decision = gate.decide(\'retrained_AND\', json.dumps(proposal), None, 1, 6,\n        \'Is the next assistant action supported by the public fact?\', [\'Paris is the capital of France.\'])\n    metadata = json.loads((out/\'runtime.json\').read_text())\n    validate_health(metadata[\'writer\'])\n    assert metadata[\'writer\'][\'model\'] == MODEL\n    assert isinstance(decision[\'p_approve\'], float)\n    receipt = dict(status=\'passed\', kind=\'invented engineering input, not official Tau task success\',\n        proposal=proposal, review=review, gate=decision, usage=agents.usage, runtime=metadata)\n    with (out/\'qualification.json\').open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(receipt, stream, indent=2)\n    print(json.dumps(receipt, indent=2), flush=True)\n'
import signal
engineering_log = OUT/('gpu-smoke-7b' + "-console.log")
print("Engineering output is being preserved in", engineering_log, flush=True)
with engineering_log.open("x", encoding="utf-8") as log:
    process = subprocess.Popen([sys.executable, "-u", "-B", "-c", engineering_script,
        str(OUT/'gpu-smoke-7b'), str(HEAD)], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
        env=dict(os.environ, LOGURU_LEVEL="ERROR", TOKENIZERS_PARALLELISM="false"))
    try:
        process.wait()
    except BaseException:
        if process.poll() is None:
            # SIGINT gives the owned runtime its context-manager cleanup path.
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        log.flush()
        print(engineering_log.read_text(encoding="utf-8")[-10000:], flush=True)
        raise
print(engineering_log.read_text(encoding="utf-8")[-10000:], flush=True)
if process.returncode:
    raise RuntimeError("Engineering qualification failed; complete output preserved at " + str(engineering_log))


## 6. Qualify full-context capacity
This required invented stress input preserves exactly 15,300 prompt tokens and exercises 768 output tokens with the critic resident. GPU memory is sampled. An early EOS, missing samples or any error fails qualification; preserve that evidence and inspect it before continuing.

In [ ]:
assert json.loads((OUT/"gpu-smoke-7b/qualification.json").read_text())["status"] == "passed"
engineering_script = 'import json, subprocess, sys, threading, time\nfrom pathlib import Path\nfrom transformers import AutoTokenizer\nfrom openai import OpenAI\nfrom haltiq_pair.core import Agents\nfrom haltiq_pair.strong_writer import StrongRuntime, MODEL, REVISION, validate_health\nfrom haltiq_pair.team_training import StrandsHead\nout, head_path = Path(sys.argv[1]), Path(sys.argv[2])\nassert not out.exists(), "Preserve earlier engineering qualifications"\nwith StrongRuntime(out, StrandsHead.from_json(head_path)) as (agents, gate):\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)\n    system = \'This is an invented capacity test. Continue repeating only the word moss separated by spaces indefinitely. Do not finish voluntarily, explain, or add other words.\'\n    def prompt_for(count):\n        return \'moss \' * count + \'\\nContinue now with only moss repeated indefinitely, without stopping.\'\n    def token_count(prompt):\n        return len(tokenizer.apply_chat_template([{\'role\':\'system\',\'content\':system},\n            {\'role\':\'user\',\'content\':prompt}], tokenize=True, return_dict=False, add_generation_prompt=True))\n    low, high = 1, 32000\n    while low < high:\n        middle = (low + high + 1)//2\n        if token_count(prompt_for(middle)) <= 15300: low = middle\n        else: high = middle - 1\n    prompt = prompt_for(low)\n    expected = token_count(prompt)\n    assert expected == 15300, (\'Exact stress-input length not reached\', expected)\n    samples, monitor_errors, done = [], [], threading.Event()\n    def monitor():\n        while not done.is_set():\n            try:\n                value = subprocess.check_output([\'nvidia-smi\',\'--query-gpu=memory.used\',\n                    \'--format=csv,noheader,nounits\'], text=True, timeout=5)\n                samples.append(int(value.strip().splitlines()[0]))\n            except Exception as error:\n                monitor_errors.append(type(error).__name__)\n            done.wait(.5)\n    thread = threading.Thread(target=monitor, daemon=True)\n    thread.start()\n    started = time.monotonic()\n    try:\n        assert gate.critic is not None\n        client = OpenAI(api_key=\'local\', base_url=\'http://127.0.0.1:8001/v1\', timeout=600, max_retries=0)\n        response = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=768,\n            messages=[{\'role\':\'system\',\'content\':[{\'type\':\'text\',\'text\':system}]},\n                      {\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':prompt}]}])\n        actual = dict(prompt_tokens=response.usage.prompt_tokens,\n            completion_tokens=response.usage.completion_tokens,total_tokens=response.usage.total_tokens)\n        assert actual[\'prompt_tokens\'] == expected, (\'Prompt was changed or truncated\', actual, expected)\n        assert actual[\'completion_tokens\'] == 768, (\'Full decode stress was not reached\', actual)\n        short = Agents(model_id=MODEL, max_tokens=64)\n        short_output = short.generate(\'writer\',\'Return only the capital name.\',\n            \'Paris is the capital of France. What is its capital?\')\n        decision = gate.decide(\'retrained_AND\',\'Paris\',None,1,6,\n            \'What is the capital of France?\',[\'France: Paris is its capital.\'])\n        assert isinstance(decision[\'p_approve\'], float)\n    finally:\n        done.set()\n        thread.join(timeout=10)\n    metadata = json.loads((out/\'runtime.json\').read_text())\n    validate_health(metadata[\'writer\'])\n    assert metadata[\'writer\'][\'prefill_chunk_size\'] == 512\n    assert metadata[\'writer\'][\'max_context\'] == 16384\n    assert samples and not monitor_errors\n    receipt = dict(status=\'passed\', kind=\'invented engineering qualification, not benchmark performance\',\n        expected_prompt_tokens=expected, actual_usage=actual, generation_cap=768,\n        complete_prompt_preserved=True, full_decode_budget_exercised=True,\n        critic_resident_during_writer=True, peak_sampled_total_gpu_mib=max(samples),\n        peak_scope=\'sampled during stress inference, not model-loading peak\',\n        gpu_memory_samples=len(samples),monitor_errors=monitor_errors,\n        seconds=time.monotonic()-started, stress_finish_reason=response.choices[0].finish_reason,\n        stress_output=response.choices[0].message.content,post_stress_sdk_output=short_output,\n        post_stress_gate=decision, runtime=metadata)\n    with (out/\'qualification.json\').open(\'x\',encoding=\'utf-8\') as stream:\n        json.dump(receipt,stream,indent=2)\n    print(json.dumps(receipt,indent=2),flush=True)\nprint(\'YASH_7B_LONG_CONTEXT_QUALIFIED\',flush=True)\n'
import signal
engineering_log = OUT/('gpu-long-context-7b' + "-console.log")
print("Engineering output is being preserved in", engineering_log, flush=True)
with engineering_log.open("x", encoding="utf-8") as log:
    process = subprocess.Popen([sys.executable, "-u", "-B", "-c", engineering_script,
        str(OUT/'gpu-long-context-7b'), str(HEAD)], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
        env=dict(os.environ, LOGURU_LEVEL="ERROR", TOKENIZERS_PARALLELISM="false"))
    try:
        process.wait()
    except BaseException:
        if process.poll() is None:
            # SIGINT gives the owned runtime its context-manager cleanup path.
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        log.flush()
        print(engineering_log.read_text(encoding="utf-8")[-10000:], flush=True)
        raise
print(engineering_log.read_text(encoding="utf-8")[-10000:], flush=True)
if process.returncode:
    raise RuntimeError("Engineering qualification failed; complete output preserved at " + str(engineering_log))


## 7. Run the separate official development smoke
Two public-hash-selected tasks × four policies. These eight attempts are development evidence and are excluded from the final ten-task panel. Invalid outputs may receive exactly one logged public-contract correction. Refused actions never execute.

In [ ]:
import time
def run_stage(name, arguments, destination):
    destination = pathlib.Path(destination)
    assert not destination.exists(), "Preserve previous stage outputs; choose a new OUT"
    log_path = OUT/(name + "-console.log")
    with log_path.open("x", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, "-u", "-B", *arguments], cwd=ROOT,
            stdout=log, stderr=subprocess.STDOUT,
            env=dict(os.environ, LOGURU_LEVEL="ERROR", TOKENIZERS_PARALLELISM="false"))
        previous = None
        try:
            while process.poll() is None:
                summary = destination/"summary.json"
                if summary.exists():
                    current = summary.read_text(encoding="utf-8")
                    if current != previous:
                        value = json.loads(current)
                        print(name, json.dumps(value.get("policies", value.get("arms", value))), flush=True)
                        previous = current
                time.sleep(5)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
            raise
        if process.returncode:
            print(log_path.read_text(encoding="utf-8")[-8000:])
            raise RuntimeError(f"{name} exited {process.returncode}; preserved log: {log_path}")
    summary = json.loads((destination/"summary.json").read_text(encoding="utf-8"))
    assert summary["complete"], "Stage is incomplete; preserve evidence and inspect its logs"
    print(name, "finished; failures remain in all planned denominators.", flush=True)
    return summary
qualification = json.loads((OUT/"gpu-long-context-7b/qualification.json").read_text())
assert qualification["status"] == "passed" and qualification["full_decode_budget_exercised"]
TAU_SMOKE = OUT/"tau-development-2"
development = run_stage("tau-development-2", ["-m", "haltiq_pair.aligned_tau",
    "--head", str(HEAD), "--output", str(TAU_SMOKE), "--upstream-root", str(TAU_ROOT),
    "--phase", "smoke", "--limit", "2", "--seed", "43",
    "--taskids-manifest", str(FROZEN/"Yash-Tau-Smoke-IDs.json")], TAU_SMOKE)


## 8. Run the fresh official evaluation
Ten different retail tasks × four policies. Keep all forty attempts, failures and refusals. Do not replace tasks based on their outcomes. Local-user/judge conditions and QA-to-retail transfer limits remain explicit.

In [ ]:
assert json.loads((OUT/"tau-development-2/summary.json").read_text())["complete"]
assert json.loads((OUT/"gpu-long-context-7b/qualification.json").read_text())["status"] == "passed"
TAU_EVAL = OUT/"tau-evaluation-10"
evaluation = run_stage("tau-evaluation-10", ["-m", "haltiq_pair.aligned_tau",
    "--head", str(HEAD), "--output", str(TAU_EVAL), "--upstream-root", str(TAU_ROOT),
    "--phase", "evaluation", "--limit", "10", "--seed", "43",
    "--taskids-manifest", str(FROZEN/"Yash-Tau-Evaluation-IDs.json")], TAU_EVAL)


## Export Yash's complete evidence
The ZIP keeps frozen inputs, manifests, exact runtime receipts, primary response JSONs, traces, outcomes, summaries and logs. Critic-cache weights and Python bytecode are excluded without deleting them. Preserve incomplete and negative runs. The ZIP is created outside OUT and never overwrites another archive.

In [ ]:
import zipfile, hashlib
from google.colab import files
def create_evidence_zip(result_directory, archive_path):
    result_directory, archive_path = pathlib.Path(result_directory).resolve(), pathlib.Path(archive_path)
    assert not archive_path.resolve().is_relative_to(result_directory), "ZIP must be outside OUT"
    serving_receipts = {"hobson_config.json", "haltiq-serving-provenance.json"}
    with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as evidence:
        for path in sorted(result_directory.rglob("*")):
            relative = path.relative_to(result_directory)
            if not path.is_file() or "__pycache__" in relative.parts:
                continue
            if "critic-cache" in relative.parts and path.name not in serving_receipts:
                continue
            evidence.write(path, arcname=relative.as_posix())
    return archive_path
archive = create_evidence_zip(OUT, "/content/Yash_Team_Aligned_" + OUT.parent.name + "_" + RUN_ID + ".zip")
print("Evidence ZIP:", archive, "bytes:", archive.stat().st_size,
      "SHA256:", hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
if not storage_metadata["persistent"]:
    print("Keep the downloaded ZIP: this OUT is temporary runtime storage.")
